## End-to-End Data Encryption Pipeline Simulation

This notebook demonstrates a simulated end-to-end encryption pipeline. In a typical scenario:

1.  **Client-side (e.g., Web Browser):** Wearable JSON data is encrypted using JavaScript libraries like CryptoJS.
2.  **Server-side (e.g., Python Backend/Database):** The encrypted data (ciphertext) is received and stored in a database.
3.  **Server-side (for processing):** Encrypted data is retrieved from the database and decrypted for analysis or display.

Since this Colab environment runs Python, we will use Python's `cryptography` library to *simulate* the encryption and decryption steps. In a real application, the encryption part would typically happen in the client's browser with CryptoJS, and the decryption would happen on a trusted server (which could be Python-based).

**Security Note:** For demonstration purposes, the encryption key will be generated and used within the same notebook. In a production environment, keys must be securely managed (e.g., using a Key Management Service) and never hardcoded or exposed.

In [1]:
# Install the necessary library
!pip install cryptography

In [2]:
import json
import base64
from cryptography.fernet import Fernet

# --- 1. Key Generation (In a real scenario, this key would be securely managed) ---
# Generate a new encryption key (for demonstration)
# In a real system, this key would be generated once and stored securely.
encryption_key = Fernet.generate_key()
fernet = Fernet(encryption_key)

print("Generated Encryption Key (Base64 encoded):")
print(encryption_key.decode())
print("\n--- Encryption and Decryption Functions ---")

Generated Encryption Key (Base64 encoded):
fF8ir2ObSc3LsnUymqUIk0IB2khZD4sVkLDsczU6YL8=

--- Encryption and Decryption Functions ---


In [3]:
def encrypt_data(data: dict) -> str:
    """Encrypts JSON data using Fernet and returns a base64 encoded string."""
    # Convert dict to JSON string
    json_string = json.dumps(data)
    # Encode the JSON string to bytes
    json_bytes = json_string.encode('utf-8')
    # Encrypt the bytes
    encrypted_bytes = fernet.encrypt(json_bytes)
    # Return base64 encoded string for storage
    return base64.urlsafe_b64encode(encrypted_bytes).decode('utf-8')

def decrypt_data(encrypted_str: str) -> dict:
    """Decrypts a base64 encoded string and returns the original JSON data."""
    # Decode the base64 string to bytes
    encrypted_bytes = base64.urlsafe_b64decode(encrypted_str)
    # Decrypt the bytes
    decrypted_bytes = fernet.decrypt(encrypted_bytes)
    # Decode bytes to JSON string
    json_string = decrypted_bytes.decode('utf-8')
    # Convert JSON string to dict
    return json.loads(json_string)


In [4]:
# --- 2. Simulate Incoming Wearable JSON Data ---
# Example of raw wearable data
wearable_data = {
    "user_id": "user123",
    "device_id": "watch_alpha",
    "timestamp": "2023-10-27T10:30:00Z",
    "heart_rate": 72,
    "steps": 1200,
    "location": {
        "latitude": 34.0522,
        "longitude": -118.2437
    },
    "activity": "walking"
}

print("\n--- Original Wearable JSON Data ---")
print(json.dumps(wearable_data, indent=2))


--- Original Wearable JSON Data ---
{
  "user_id": "user123",
  "device_id": "watch_alpha",
  "timestamp": "2023-10-27T10:30:00Z",
  "heart_rate": 72,
  "steps": 1200,
  "location": {
    "latitude": 34.0522,
    "longitude": -118.2437
  },
  "activity": "walking"
}


In [5]:
# --- 3. Encrypt Data and Simulate Database Storage ---

# Encrypt the data
encrypted_payload = encrypt_data(wearable_data)

# Simulate storing the ciphertext in a 'database'
# In a real application, this would be a database table storing ciphertexts.
database_records = []
database_records.append({
    "record_id": "rec_001",
    "encrypted_data": encrypted_payload,
    "encryption_timestamp": "2023-10-27T10:31:00Z" # Metadata
})

print("\n--- Encrypted Data (Ciphertext) Stored in 'DB' ---")
print(json.dumps(database_records[0], indent=2))
print(f"\nLength of original data: {len(json.dumps(wearable_data))} bytes")
print(f"Length of encrypted data: {len(encrypted_payload)} bytes")


--- Encrypted Data (Ciphertext) Stored in 'DB' ---
{
  "record_id": "rec_001",
  "encrypted_data": "Z0FBQUFBQnA0R25qek5HTGR5UkpkREktTjBveWxmR2xiZTkxLVcxTGhFRTExSjZVYXM4cng0QnVFZjljdlRnXy1ZczkzQl94UFVJQmVVY0dnd1ZHQ2VCdHI3LUJfSGY5U1RUVk5yRl9vcXhISW1qaW4xY1JXUTZEMzQ4S2drMHdISHpCWjV1aTlkTHc5SVNTSmtRcGFFU01GWEVfbDRianoyLThqWURCMHRzRkRCOVJkVDJZc3J4VzlUNjd0OVRCRWg3V05sM0lkbE8wcW9zZG1oM2gyVnY1U3NSM2ZyZlR3blprVjgyNUE5em5laWhrSjZzQzVLTmZQQXhrSVhUanBCb1BqLUk4T3FzelA5VndhNjdRakZ1V3A4ODhKOWxXenB3X254d1dFejhEcVJmTUU1Y3diTGxaSTQtUXkzOUVEWkRJWmgzcExOcmdnV040ZkRremljbC0zN1F6cGt3cUpRPT0=",
  "encryption_timestamp": "2023-10-27T10:31:00Z"
}

Length of original data: 202 bytes
Length of encrypted data: 476 bytes


In [6]:
# --- 4. Simulate Retrieving from 'DB' and Decrypting ---

# Simulate retrieving an encrypted record from the 'database'
retrieved_record = database_records[0]
retrieved_encrypted_data = retrieved_record['encrypted_data']

print("\n--- Retrieving and Decrypting Data ---")
print(f"Retrieved encrypted data: {retrieved_encrypted_data[:50]}...\n")

# Decrypt the data
decrypted_data = decrypt_data(retrieved_encrypted_data)

print("--- Decrypted Data ---")
print(json.dumps(decrypted_data, indent=2))

# Verify if the decrypted data matches the original
assert decrypted_data == wearable_data
print("\nVerification successful: Decrypted data matches original data.")


--- Retrieving and Decrypting Data ---
Retrieved encrypted data: Z0FBQUFBQnA0R25qek5HTGR5UkpkREktTjBveWxmR2xiZTkxLV...

--- Decrypted Data ---
{
  "user_id": "user123",
  "device_id": "watch_alpha",
  "timestamp": "2023-10-27T10:30:00Z",
  "heart_rate": 72,
  "steps": 1200,
  "location": {
    "latitude": 34.0522,
    "longitude": -118.2437
  },
  "activity": "walking"
}

Verification successful: Decrypted data matches original data.


## Real-world Context with CryptoJS

In a real-world end-to-end encryption pipeline involving client-side wearable data and a backend:

1.  **Client (Wearable Device/Companion App):** The wearable device or its companion mobile/web app would use JavaScript (or a similar client-side language) with libraries like **CryptoJS** to encrypt the JSON data *before* sending it over the network.
    ```javascript
    // Example using CryptoJS (conceptual client-side code)
    // var encrypted = CryptoJS.AES.encrypt(JSON.stringify(wearableData), encryptionKey.toString()).toString();
    // Send 'encrypted' string to backend
    ```

2.  **Backend (Python/Colab Simulation):** The Python backend (like this Colab notebook) would receive this already-encrypted data. It would then store this ciphertext in a database without ever seeing the plaintext. When processing is required, the backend would retrieve the ciphertext and use a compatible decryption mechanism (e.g., a Python library like `cryptography` configured with the same algorithm and key derivation as CryptoJS) to decrypt the data.

The key management is crucial: the encryption key used by CryptoJS on the client-side must either be derived from a shared secret, or securely transmitted to the server for decryption. Often, hybrid approaches (e.g., using asymmetric encryption for key exchange) are used for robust key management.